In [2]:
import numpy as np
import pandas as pd

# text preprocessing
import re
import swifter
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# training fasttext model
from gensim.models import FastText
from gensim.models.fasttext import load_facebook_model

tip_file_path = "/kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_tip.json"

## Step 0: Preproccsing
- read `text` column from `yelp_academic_dataset_tip`
- import the `english stop words` and `WordNetLemmatizer`
- preprocess text using `preprocess` function

In [3]:
def preprocess(text: str):
    text = text.lower()
    
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    tokens = text.split()
    tokens = [word for word in tokens if len(word) > 3]

    lemma_text = [lemmatizer.lemmatize(word) for word in tokens]
    clean_tokens = [word for word in lemma_text if word not in en_stop]
    
    return clean_tokens

In [6]:
df = pd.read_json(
    tip_file_path,
    lines=True,
).text

print(df.shape)
print(df.head())

(908915,)
0                       Avengers time with the ladies.
1    They have lots of good deserts and tasty cuban...
2               It's open even when you think it isn't
3                            Very decent fried chicken
4               Appetizers.. platter special for lunch
Name: text, dtype: object


In [18]:
en_stop = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [19]:
corpus = df.swifter.apply(preprocess).tolist()

Pandas Apply:   0%|          | 0/908915 [00:00<?, ?it/s]

## Step 1: Custom Model Training and Testing
- train fasttext model
- Print 10 similar words & 10 opposite words for a given/input words

In [32]:
model = FastText(vector_size=100, window=5, min_count=1, workers=4, sg=1)
model.build_vocab(corpus)

In [33]:
model.train(corpus, total_examples=model.corpus_count, epochs=10)

(47081262, 53946600)

In [34]:
test_word = 'strong'

In [35]:
# Check top 10 similar word for a given word by gensim fastText
for word, sim in model.wv.most_similar(test_word, topn=10):
    print(f'{word}: {sim}')

trong: 0.8511404395103455
stronger: 0.8259915113449097
stronghold: 0.8237668871879578
sweetbutstrong: 0.7938825488090515
strongbows: 0.7863648533821106
strongberri: 0.7767446041107178
strongest: 0.7729789018630981
strongbow: 0.7621766328811646
strrrong: 0.7350889444351196
weakly: 0.7156550884246826


In [36]:
# Most opposite to a word
for word, sim in model.wv.most_similar(negative=[test_word], topn=10):
    print(f'{word}: {sim}')

yqqm: 0.12117426097393036
日料好赞: 0.05599500983953476
日本やん: 0.0418294221162796
絶対お勧め: -0.014065186493098736
料好實在: -0.018589433282613754
チェックインオファーあり: -0.028567150235176086
口味選擇多: -0.040408145636320114
html: -0.05090619623661041
yessssssssssssssssssssssssssssssssssssss: -0.052103932946920395
http: -0.05856046453118324


## Step 2: Test pretrained model
- Print 10 similar words & 10 opposite words for a given/input words

In [23]:
pretrained_model_path = "/kaggle/input/models/ahmedabdulhakeem/pretrainedfasttextmodel/other/default/1/cc.en.300.bin"
pretrained_model = load_facebook_model(pretrained_model_path)

In [37]:
# Check top 10 similar word for a given word by gensim fastText
for word, sim in pretrained_model.wv.most_similar(test_word, topn=10):
    print(f'{word}: {sim}')

strong.So: 0.9921470880508423
strong.But: 0.9917914271354675
strong.This: 0.9910361170768738
strong.If: 0.9909799098968506
strong.As: 0.9909595251083374
strong.A: 0.99091637134552
strong.The: 0.9907025098800659
strong.And: 0.990568995475769
not-so-strong: 0.9901255369186401
strong.When: 0.990106463432312


In [38]:
for word, sim in pretrained_model.wv.most_similar(negative=[test_word], topn=10):
    print(f'{word}: {sim}')

.4.4: 0.15560497343540192
Page6: 0.15405289828777313
Page7: 0.15049204230308533
COMMUNIQUÉ: 0.14893855154514313
17273: 0.14203830063343048
19691: 0.13590370118618011
Page9: 0.13528484106063843
LLC4: 0.13419197499752045
15939: 0.13336966931819916
17442: 0.13045242428779602


## Step 3: Update the pretrained model using our corpus
- Print 10 similar words & 10 opposite words for a given/input words

In [26]:
len(corpus)

908915

In [27]:
pretrained_model.build_vocab(corpus, update=True)

In [29]:
pretrained_model.train(corpus, total_examples=len(corpus), epochs=10)

(13718831, 53946600)

In [39]:
# Check top 10 similar word for a given word by gensim fastText
for word, sim in pretrained_model.wv.most_similar(test_word, topn=10):
    print(f'{word}: {sim}')

strong.So: 0.9921470880508423
strong.But: 0.9917914271354675
strong.This: 0.9910361170768738
strong.If: 0.9909799098968506
strong.As: 0.9909595251083374
strong.A: 0.99091637134552
strong.The: 0.9907025098800659
strong.And: 0.990568995475769
not-so-strong: 0.9901255369186401
strong.When: 0.990106463432312


In [40]:
for word, sim in pretrained_model.wv.most_similar(negative=[test_word], topn=10):
    print(f'{word}: {sim}')

.4.4: 0.15560497343540192
Page6: 0.15405289828777313
Page7: 0.15049204230308533
COMMUNIQUÉ: 0.14893855154514313
17273: 0.14203830063343048
19691: 0.13590370118618011
Page9: 0.13528484106063843
LLC4: 0.13419197499752045
15939: 0.13336966931819916
17442: 0.13045242428779602
